# CAP F1 Abstention
This notebook (hackily) adapts Jimin's CAP F1 code to use with our self-consistency based absention work.

## Include Library

In [ ]:
from datetime import datetime
import os
import random

# library for cap_f1
from cap_f1 import LLMClient, AtomicProcessor, ResultsRepo
from fewshot_examples import (
    FEWSHOT_DEDUP_MESSAGES,
    FEWSHOT_RECALL_MESSAGES,
    FEWSHOT_PRECISION_MESSAGES,
)

# code for no need for restarting the kernel when python file is updated
%load_ext autoreload
%autoreload 2

## 1. build API + processor (inject few-shot examples if you want)
currently fewshot example is at fewshot_examples.py

In [ ]:
# 1) build API + processor (inject few-shot examples if you want)
llm = LLMClient()
proc = AtomicProcessor(
    llm,
    fewshot_dedup=FEWSHOT_DEDUP_MESSAGES,  # or None
    fewshot_recall=FEWSHOT_RECALL_MESSAGES,  # or None
    fewshot_precision=FEWSHOT_PRECISION_MESSAGES,  # or None
)

## 2. Load Data

In [ ]:
def get_random_sample(dataset, count=1):
    """Get a random sample from the dataset

    Args:
        dataset (list): List of dictionaries
        count (int, optional): Number of samples to return. Defaults to 1.

    Returns:
        list: List of dictionaries (random samples)
    """
    return random.sample(dataset, count)

In [ ]:
print("Loading abstention dataset...")

# number of data points testing
LIMIT = 50

# for filename
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M")

# create folder to save the results
folder_path = f"results/{timestamp}"
os.makedirs(folder_path, exist_ok=True)

# features that we need to extract from the original dataset
org_caption_dataset = ResultsRepo.read_json(
    "test-data-scored-with-mt-metrics_300-examples_2025-08-12_12-29-55.json"
)
# org_caption_dataset = org_caption_dataset[:LIMIT]
org_caption_dataset = get_random_sample(org_caption_dataset, count=LIMIT)
org_caption_dataset

### Hacks for getting abstention data to work with current CAP F1

1. Change human_captions to human_captions_crowdworkers
2. Create a new human_captions field with the following:

```python
'human_captions': [
    {
        'caption': <greedy caption goes here>,
        'is_precanned': False,
        'is_rejected': False
    }
]
```
3. Create a model_captions to hold samples as:
```python
'model_captions': [
    {
        'model_name': 'sample_1',
        'caption': ''
    },
    {
        'model_name': 'sample_2',
        'caption': ''
    },
    ...
    {
        'model_name': 'sample_10',
        'caption': ''
    },
],
```

In [ ]:
for item in org_caption_dataset:
    # hack 1
    item["crowdworker_captions"] = item["human_captions"]
    del item["human_captions"]

    # hack 2
    item["human_captions"] = [
        {
            "caption": item["greedy_response"],
            "is_precanned": False,
            "is_rejected": False,
        }
    ]

    # hack 3
    item["model_captions"] = [
        {
            "model_name": f"sample_{index + 1}",
            "caption": sample,
        }
        for index, sample in enumerate(item["additional_responses"])
    ]

    # add an evaluation field
    item["evaluation"] = {}
org_caption_dataset[0]

### 3. generate atomics

In [ ]:
# 3) generate atomics
print(f"Generating atomic statements using {llm.model}")
T_atomics, g_atomics, parsed_T = proc.generate_atomic_statement(
    org_caption_dataset, limit=LIMIT
)

# 3.1) save intermediate
print("Saving intermediate results...")
all_human_captions = []
for item in org_caption_dataset:
    # Filter out human captions that are mention quality issues
    human_captions = [
        hc["caption"]
        for hc in item["human_captions"]
        if hc["caption"] != "Quality issues are too severe to recognize visual content."
    ]
    all_human_captions.append(human_captions)
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/intermediate_{timestamp}.json",
    org_dataset=org_caption_dataset,
    T_atomics=T_atomics,
    g_atomics=g_atomics,
    parsed_T=parsed_T,
    T_org=all_human_captions,
    limit=LIMIT,
)

### 4. evaluate and get recall and precision
- match human caption to model caption
- create recall and precision data

In [ ]:
# before calculating F1 score, match sentences between human generated and model generated
print("Evaluating atomic statements...")
eval_out = proc.evaluate_matching(all_human_captions, T_atomics, g_atomics)

# 4.1) save evaluation results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/eval_{timestamp}.json",
    update_existing=f"{folder_path}/intermediate_{timestamp}.json",
    metadata=eval_out,
    limit=LIMIT,
)

### 5. calculate cap f1 score


In [ ]:
# 5) calculate cap f1 score
cap_scores = proc.calculate_cap_f1(eval_out)

# 5.1) save cap f1 score results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/final_{timestamp}.json",
    update_existing=f"{folder_path}/eval_{timestamp}.json",
    evaluations=cap_scores,
    limit=LIMIT,
)

In [ ]:
# 6) Final JSON → CSV
print("Saving final results into csv...")
ResultsRepo.export_final_csv(
    json_path=f"{folder_path}/final_{timestamp}.json",
    csv_path=f"{folder_path}/final_{timestamp}.csv",
    # model_keys={"gpt":"gpt-4o-2024-08-06", "molmo":"Molmo-7B-O-0924", "llama":"Llama-3.2-11B-Vision-Instruct"}
)

### Export as formatted CSV

In [ ]:
# import json
import json
import pandas as pd

output_data = json.load(open(f"{folder_path}/eval_{timestamp}.json"))

min_f1 = 0.0
max_f1 = 0.0
for output in output_data:
    keys_to_remove = [
        "expected_match_count_scores",
        "match_count_scores",
        "greedy_response_no_stop",
        "additional_responses_no_stop",
        "bleu-1",
        "bleu-2",
        "bleu-3",
        "bleu-4",
        "meteor",
        "rouge",
        "cider",
        "spice",
        "bertscore",
        "bertscore_idf",
        "crowdworker_captions",
        "human_captions",
        "model_captions",
    ]

    for key in keys_to_remove:
        del output[key]

    # expand additionaal_responses into separate keys as sample_1 .. sample_n
    additional_responses = output["additional_responses"]
    for i, response in enumerate(additional_responses):
        output[f"sample_{i + 1}"] = response
    del output["additional_responses"]

    # get t_atomics
    output["t_atomics"] = output["evaluation"]["cap_f1"]["T_atomics"]

    # get g_atomics for each sample
    all_g_atomics = output["evaluation"]["cap_f1"]["g_atomics"]
    for key in all_g_atomics.keys():
        output[f"{key}_g_atomics"] = all_g_atomics[key]

    # get all recall and precisions scores
    all_scores = output["evaluation"]["cap_f1"]["scores"]
    all_recall = []
    all_precision = []
    for key in all_scores.keys():
        output[f"{key}_recall"] = all_scores[key]["recall"]
        output[f"{key}_precision"] = all_scores[key]["precision"]

        all_recall.append(all_scores[key]["recall"])
        all_precision.append(all_scores[key]["precision"])

    # compute average recall and precision
    output["average_recall"] = sum(all_recall) / len(all_recall)
    output["average_precision"] = sum(all_precision) / len(all_precision)

    # compute average f1 as the harmonic mean of recall and precision
    output["average_f1"] = (
        2
        * (output["average_recall"] * output["average_precision"])
        / (output["average_recall"] + output["average_precision"])
    )

    min_f1 = min(min_f1, output["average_f1"])
    max_f1 = max(max_f1, output["average_f1"])

# normalize f1 score
all_f1 = []
for output in output_data:
    output["normalized_f1"] = (output["average_f1"] - min_f1) / (max_f1 - min_f1)

# save as csv
pd.DataFrame(output_data).to_csv(
    "./results/2025-08-20_23-23/final_2025-08-20_23-23_cleaned.csv", index=False
)